# Lab 9 - Magentic orchestration

## What will you do?

Labs 7 and 8 used a fixed pipeline and a fixed fan-out. Here the order of work is not fixed at all: a **manager agent** builds and adapts a task list (a **task ledger**) as it consults specialist agents, using Agent Framework's **`MagenticBuilder`**.

![Diagram that shows magentic orchestration.](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/_images/magentic-pattern.svg)

*Magentic orchestration. Source: [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#magentic-orchestration) on Microsoft Learn.*

The task here has no predetermined solution path: plan a community outreach push to raise flu-vaccination uptake before winter. The manager decides which specialist to consult and in what order, then synthesizes their input into one plan.

> **This is a workshop exercise, not a clinical tool.**

## Before you start

- Python 3.11 or later, with a notebook kernel selected, and `az login` completed.
- Lab 1 finished: a project endpoint and an approved model deployment.
- No knowledge base or tools are needed for this lab.

Install the pinned package set below. If you already imported a different version of these SDKs in this kernel, restart the kernel after installing.

In [ ]:
%pip install -q "azure-ai-projects==2.3.0" "azure-identity==1.25.3" "openai==2.54.0" "agent-framework-core==1.16.0" "agent-framework-openai==1.14.1" "agent-framework-foundry==1.11.0" "agent-framework-orchestrations==1.1.1"

## 0. Connect to your project

Use your Lab 1 settings. The cell signs in with your Azure CLI account and generates a suffix to keep your agents unique in the shared project.

In [ ]:
import asyncio
import os
import sys
from contextlib import AsyncExitStack
from uuid import uuid4

from agent_framework.foundry import FoundryAgent
from agent_framework.orchestrations import MagenticBuilder
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import AzureCliCredential

# Your nonsecret settings from Lab 1. Paste them between the quotes, or set them as
# environment variables before starting the kernel.
PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")

if sys.version_info < (3, 11):
    raise RuntimeError(
        "These notebooks need Python 3.11 or later. This kernel is "
        f"{sys.version_info.major}.{sys.version_info.minor}. Select a newer kernel."
    )

missing = [
    name
    for name, value in {
        "AZURE_AI_PROJECT_ENDPOINT": PROJECT_ENDPOINT,
        "AZURE_AI_MODEL_DEPLOYMENT_NAME": MODEL_DEPLOYMENT,
    }.items()
    if not value
]
if missing:
    raise ValueError(f"Set these before continuing: {', '.join(missing)}")


def check_todos(**answers: object) -> None:
    """Helper. Stops the cell while a `...` blank is still open."""
    still_open = [name for name, value in answers.items() if value is ...]
    if still_open:
        raise ValueError(f"Fill in these blanks first: {', '.join(still_open)}")


SUFFIX = uuid4().hex[:8]
credential = AzureCliCredential()
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
print(f"Setup complete. Suffix: {SUFFIX}")

## 1. Register a manager and two specialists

The manager does not do the outreach research or writing itself; it plans, delegates to the specialists below, and decides when the task is done.

| | Public-health researcher | Communications strategist |
|---|---|---|
| Focus | Which groups have low uptake, and why | How to reach those groups and what to say |

### To-Do 1 - Write the two specialist instructions

**Goal:** two specialists with clearly different jobs for the manager to delegate between.

**Steps**

1. Write `RESEARCHER_INSTRUCTIONS` and `COMMUNICATOR_INSTRUCTIONS`.
2. Run the cell.

**Run the cell. You should see** three agent names printed: manager, researcher and communicator.

<details><summary>Hint</summary>

The researcher identifies gaps and reasons; the communicator turns that into an outreach message and channel choice.

</details>

<details><summary>Show solution code</summary>

```python
RESEARCHER_INSTRUCTIONS = (
    "You identify which community groups likely have low flu-vaccination uptake and why, "
    "using general public-health reasoning."
)
COMMUNICATOR_INSTRUCTIONS = (
    "You turn the researcher's findings into concrete outreach actions: channel, message and timing."
)
```

</details>

In [ ]:
MANAGER_INSTRUCTIONS = "You coordinate a small team to produce one clear, actionable outreach plan."
RESEARCHER_INSTRUCTIONS = ...  # TODO 1: identify low-uptake groups and why.
COMMUNICATOR_INSTRUCTIONS = ...  # TODO 1: turn findings into outreach actions.
check_todos(RESEARCHER_INSTRUCTIONS=RESEARCHER_INSTRUCTIONS, COMMUNICATOR_INSTRUCTIONS=COMMUNICATOR_INSTRUCTIONS)

manager_version = project.agents.create_version(
    agent_name=f"day2-outreach-manager-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=MANAGER_INSTRUCTIONS),
)
researcher_version = project.agents.create_version(
    agent_name=f"day2-outreach-researcher-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=RESEARCHER_INSTRUCTIONS),
)
communicator_version = project.agents.create_version(
    agent_name=f"day2-outreach-communicator-{SUFFIX}",
    definition=PromptAgentDefinition(model=MODEL_DEPLOYMENT, instructions=COMMUNICATOR_INSTRUCTIONS),
)
print("Registered:", manager_version.name, researcher_version.name, communicator_version.name)

## 2. Build and run the Magentic workflow

`MagenticBuilder` takes a `manager_agent` plus a list of specialist `participants`. The manager decides who to consult, for how many rounds, before it produces a final answer. `max_round_count`, `max_stall_count` and `max_reset_count` bound how long that can take.

In [ ]:
TASK = (
    "Plan a two-week community outreach push to raise flu-vaccination uptake before winter. "
    "Recommend concrete actions, and note which ones need budget or clinical staff time."
)

async with AsyncExitStack() as stack:
    manager = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=manager_version.name,
            agent_version=str(manager_version.version),
            credential=credential,
            name="manager",
            allow_preview=False,
            timeout=240,
        )
    )
    researcher = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=researcher_version.name,
            agent_version=str(researcher_version.version),
            credential=credential,
            name="researcher",
            allow_preview=False,
            timeout=240,
        )
    )
    communicator = await stack.enter_async_context(
        FoundryAgent(
            project_endpoint=PROJECT_ENDPOINT,
            agent_name=communicator_version.name,
            agent_version=str(communicator_version.version),
            credential=credential,
            name="communicator",
            allow_preview=False,
            timeout=240,
        )
    )

    workflow = MagenticBuilder(
        participants=[researcher, communicator],
        manager_agent=manager,
        max_round_count=6,
        max_stall_count=2,
        max_reset_count=1,
    ).build()
    events = await asyncio.wait_for(workflow.run(TASK), timeout=900)

outputs = events.get_outputs()
final = outputs[-1]
plan_text = getattr(final, "text", None)
if plan_text is None and hasattr(final, "messages"):
    plan_text = "\n".join(message.text for message in final.messages)

print("FINAL PLAN\n")
print(plan_text)

## Deterministic success check

Magentic runs are open-ended, so this checks the shape of the run, not the wording: manager and specialists are distinct, and a non-empty plan came back.

In [ ]:
assert len({manager_version.name, researcher_version.name, communicator_version.name}) == 3, (
    "The manager and the two specialists should be distinct agents."
)
assert plan_text and plan_text.strip(), "The manager returned no synthesized plan."

print("PASS - the manager coordinated both specialists and produced a synthesized plan.")

## What you learned

1. Magentic orchestration does not fix an order. The manager agent decides, round by round, who to consult next.
2. The manager's task ledger tracks progress and decides when the task is done - `max_round_count` bounds how long that can take.
3. Prefer Sequential or Concurrent (Labs 7-8) whenever the plan of approach can be fixed ahead of time. Magentic trades more calls and latency for that flexibility.

**Further reading:** [Magentic orchestration](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/magentic) on Microsoft Learn, and [AI Agent Orchestration Patterns](https://learn.microsoft.com/en-us/azure/architecture/ai-ml/guide/ai-agent-design-patterns#magentic-orchestration) for when to choose this pattern.

**Next:** Lab 10 adds a guardrail to the agents you have built.

In [ ]:
project.close()
credential.close()
print("Closed the local clients. All Foundry agents remain.")